# Data Hub, Integracion y Gobierno del Dato

**Bloque 6 -- Gobierno del Dato y Estrategia | Sesion 2 de 5**

En la sesion anterior construimos el "plano" de una arquitectura de datos: capas, tipos, el patron Bronze-Silver-Gold. Hoy vamos a descubrir que ocurre cuando ese plano no se respeta -- o directamente no existe. Trabajaremos sobre un escenario de auditoria interna en el Hospital Santa Clara que revelara problemas reales de integracion, y a partir de ahi construiremos las soluciones: el Data Hub como patron tecnico y el gobierno del dato como marco organizativo.

Al finalizar la sesion seras capaz de:

- Identificar problemas tipicos de integracion de datos en una organizacion.
- Explicar que es un Data Hub y como se diferencia de otros patrones de integracion.
- Definir que es el gobierno del dato y cuales son sus pilares fundamentales.
- Asignar roles y responsabilidades de gobierno a situaciones concretas.
- Priorizar acciones correctivas en un plan de respuesta a una auditoria.

## 1. El problema: auditoria de datos en Santa Clara

Situemos el escenario. La direccion del Hospital General Santa Clara ha detectado discrepancias preocupantes en los informes mensuales: el numero de pacientes atendidos no cuadra entre el sistema de urgencias y el de facturacion, hay medicamentos dispensados a pacientes que supuestamente no tenian prescripcion activa, y varios informes de laboratorio estan asociados a identificadores de paciente que no existen en el HIS.

Se encarga una auditoria interna de calidad de datos. El equipo auditor cruza los registros de los tres sistemas principales (HIS, farmacia y laboratorio) y encuentra lo siguiente:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Resultados de la auditoria: datos del mismo paciente en tres sistemas distintos
his = pd.DataFrame({
    'id_paciente': ['P-1001', 'P-1002', 'P-1003', 'P-1004', 'P-1005'],
    'nombre': ['Maria Garcia Lopez', 'Juan Perez Martinez', 'Ana Ruiz Fernandez',
               'Carlos Sanchez Diaz', 'Lucia Martin Gomez'],
    'fecha_nacimiento': ['1985-03-12', '1972-08-25', '1990-11-03', '1968-05-17', '2001-01-30'],
    'alergias': ['Penicilina', 'Ninguna', 'Latex, Ibuprofeno', 'Ninguna', 'Polen']
})

farmacia = pd.DataFrame({
    'id_paciente': ['P-1001', 'P-1002', 'P-1003', 'P-1004', 'P-1005'],
    'nombre': ['Maria Garcia', 'Juan Perez Martinez', 'Ana Ruiz', 
               'Carlos Sanchez Diaz', 'Lucia Martin'],
    'fecha_nacimiento': ['1985-03-12', '1972-08-25', '1990-11-03', '1968-05-17', '2001-01-30'],
    'alergias': ['Penicilina, Amoxicilina', 'Ninguna', 'Latex', 'AAS', 'Ninguna']
})

laboratorio = pd.DataFrame({
    'id_paciente': ['P-1001', 'P-1002', 'P-1003', 'PAC-1004', 'P-1005', 'P-1099'],
    'nombre': ['M. Garcia Lopez', 'Juan Perez M.', 'Ana Ruiz Fernandez',
               'Carlos Sanchez D.', 'Lucia Martin Gomez', 'Pedro Navarro'],
    'fecha_nacimiento': ['12/03/1985', '1972-08-25', '1990-11-03', '17-05-1968', '2001-01-30', '1995-06-22'],
    'alergias': [None, None, None, None, None, None]
})

print("=== SISTEMA HIS (Historia Clinica Electronica) ===")
print(his.to_string(index=False))
print("\n=== SISTEMA FARMACIA ===")
print(farmacia.to_string(index=False))
print("\n=== SISTEMA LABORATORIO ===")
print(laboratorio.to_string(index=False))

Dediquemos un momento a examinar estos datos con ojo critico. Los problemas que revela la auditoria son representativos de lo que ocurre en organizaciones reales cuando cada sistema funciona como un silo independiente:

**Problemas de identificacion:**
- El laboratorio usa un formato de ID distinto para el paciente P-1004 ("PAC-1004" en vez de "P-1004").
- Aparece un paciente "P-1099" en laboratorio que no existe en los otros sistemas. Es un registro huerfano: o es un error de transcripcion o es un paciente que nunca se dio de alta correctamente en el HIS.

**Problemas de consistencia en nombres:**
- Maria Garcia Lopez aparece como "Maria Garcia" en farmacia y "M. Garcia Lopez" en laboratorio. Son la misma persona, pero tres representaciones distintas.
- El mismo patron se repite con varios pacientes: cada sistema almacena el nombre con su propia convencion.

**Problemas criticos de alergias:**
- HIS registra que P-1001 es alergica a Penicilina. Farmacia anade Amoxicilina (que esta en la misma familia -- correcto, pero la informacion esta incompleta en HIS).
- P-1003 tiene registradas "Latex, Ibuprofeno" en HIS pero solo "Latex" en farmacia. Si farmacia no sabe que el paciente es alergico al Ibuprofeno, podria dispensar un medicamento contraindicado.
- P-1004 no tiene alergias segun HIS, pero farmacia registra alergia a AAS (acido acetilsalicilico). Cual de los dos sistemas tiene razon?
- El laboratorio directamente no registra alergias (todos los campos son nulos). No es que los pacientes no tengan alergias; es que el sistema de laboratorio no captura esa informacion.

**Problema de formato:**
- Las fechas de nacimiento en laboratorio usan formatos inconsistentes ("12/03/1985", "17-05-1968") mientras que HIS y farmacia usan ISO 8601.

Estos no son problemas teoricos. En un entorno hospitalario, una discrepancia en el registro de alergias puede tener consecuencias clinicas graves. La pregunta clave es: **por que ocurre esto y quien es responsable de que no ocurra?**

## 2. Integracion de datos: el Data Hub como solucion tecnica

Los problemas que acabamos de ver son consecuencia directa de la falta de integracion. Cada sistema fue implementado por un equipo distinto, en momentos distintos, con criterios distintos. No hay una "fuente de verdad" que unifique la informacion del paciente.

Antes de hablar de gobierno (que es la solucion organizativa), veamos la solucion tecnica: los **patrones de integracion de datos**.

### Patron 1: Punto a punto (point-to-point)

Cada sistema se conecta directamente con los demas cuando necesita datos. Si tenemos N sistemas, necesitamos hasta N x (N-1) / 2 conexiones. Con 3 sistemas son 3 conexiones; con 10 sistemas son 45; con 20 sistemas son 190.

Es el patron mas comun en organizaciones que han crecido organicamente: cada vez que surge una necesidad se crea una conexion ad hoc. El resultado es una marana de integraciones fragiles y dificiles de mantener -- a menudo llamada "arquitectura espagueti".

### Patron 2: ETL centralizado

Un proceso centralizado extrae datos de todas las fuentes, los transforma y los carga en un destino comun (tipicamente un data warehouse). Es mas ordenado que punto a punto, pero el flujo es unidireccional: los sistemas fuente no se benefician de las correcciones realizadas durante la transformacion.

### Patron 3: Data Hub

Un nodo central que no solo almacena datos sino que los **integra, normaliza y redistribuye** a los sistemas que los necesitan. El Data Hub actua como el nucleo de una estrella: todos los sistemas se conectan a el, y el se encarga de que la informacion sea coherente en todas las direcciones.

La diferencia clave con el ETL centralizado es que el Data Hub es **bidireccional**: si farmacia detecta una nueva alergia, esa informacion fluye al hub y desde ahi se propaga al HIS y al laboratorio. No es solo un almacen; es un **orquestador de datos maestros**.

Un concepto inseparable del Data Hub es el **Master Data Management (MDM)**: la disciplina de mantener una unica version de referencia para las entidades criticas de negocio. En Santa Clara, las entidades maestras serian: paciente, profesional sanitario, medicamento, centro/servicio.

In [ ]:
# Visualizacion de los tres patrones de integracion
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Patron 1: Punto a punto ---
ax1 = axes[0]
ax1.set_xlim(-1.5, 1.5)
ax1.set_ylim(-1.5, 1.5)
ax1.set_aspect('equal')
ax1.set_title('Punto a punto\n(arquitectura espagueti)', fontsize=11, fontweight='bold')
ax1.axis('off')

sistemas_p2p = {
    'HIS':       (0, 1.1),
    'Farmacia':  (-1.1, -0.4),
    'Laborat.':  (1.1, -0.4),
    'Urgencias': (0, -1.1),
    'RRHH':      (-1.1, 0.6),
    'Facturac.': (1.1, 0.6)
}

# Dibujar todas las conexiones posibles (espagueti)
nodos = list(sistemas_p2p.values())
for i in range(len(nodos)):
    for j in range(i+1, len(nodos)):
        ax1.plot([nodos[i][0], nodos[j][0]], [nodos[i][1], nodos[j][1]],
                 'r-', alpha=0.3, linewidth=1)

for nombre, (x, y) in sistemas_p2p.items():
    ax1.plot(x, y, 'o', color='#2980b9', markersize=22)
    ax1.text(x, y, nombre, ha='center', va='center', fontsize=7, fontweight='bold', color='white')

ax1.text(0, -1.45, f'{len(nodos)}  sistemas = {len(nodos)*(len(nodos)-1)//2} conexiones',
         ha='center', fontsize=9, color='#c0392b', style='italic')

# --- Patron 2: ETL centralizado ---
ax2 = axes[1]
ax2.set_xlim(-2, 2)
ax2.set_ylim(-1.5, 1.5)
ax2.set_aspect('equal')
ax2.set_title('ETL centralizado\n(unidireccional)', fontsize=11, fontweight='bold')
ax2.axis('off')

fuentes_etl = {'HIS': (-1.5, 0.8), 'Farmacia': (-1.5, 0), 'Laborat.': (-1.5, -0.8)}
destino_etl = {'Data\nWarehouse': (1.2, 0)}

for nombre, (x, y) in fuentes_etl.items():
    ax2.plot(x, y, 'o', color='#2980b9', markersize=22)
    ax2.text(x, y, nombre, ha='center', va='center', fontsize=7, fontweight='bold', color='white')
    ax2.annotate('', xy=(0.05, 0), xytext=(x + 0.35, y),
                 arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))

# ETL box
ax2.add_patch(plt.Rectangle((-0.15, -0.35), 0.6, 0.7, fill=True,
              facecolor='#f39c12', edgecolor='#d68910', linewidth=2, zorder=5))
ax2.text(0.15, 0, 'ETL', ha='center', va='center', fontsize=9, fontweight='bold', zorder=6)

ax2.annotate('', xy=(1.0, 0), xytext=(0.55, 0),
             arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))

ax2.add_patch(plt.Rectangle((0.85, -0.35), 0.7, 0.7, fill=True,
              facecolor='#8e44ad', edgecolor='#6c3483', linewidth=2, zorder=5))
ax2.text(1.2, 0, 'Data
Warehouse', ha='center', va='center', fontsize=7,
         fontweight='bold', color='white', zorder=6)

ax2.text(0, -1.45, 'Flujo unidireccional: fuentes -> destino',
         ha='center', fontsize=9, color='#666666', style='italic')

# --- Patron 3: Data Hub ---
ax3 = axes[2]
ax3.set_xlim(-1.5, 1.5)
ax3.set_ylim(-1.5, 1.5)
ax3.set_aspect('equal')
ax3.set_title('Data Hub\n(bidireccional, orquestado)', fontsize=11, fontweight='bold')
ax3.axis('off')

sistemas_hub = {
    'HIS':       (0, 1.1),
    'Farmacia':  (-1.1, -0.4),
    'Laborat.':  (1.1, -0.4),
    'Urgencias': (0, -1.1),
    'RRHH':      (-1.1, 0.6),
    'Facturac.': (1.1, 0.6)
}

# Hub central
ax3.plot(0, 0.1, 'o', color='#27ae60', markersize=35, zorder=5)
ax3.text(0, 0.1, 'DATA
HUB', ha='center', va='center', fontsize=7,
         fontweight='bold', color='white', zorder=6)

for nombre, (x, y) in sistemas_hub.items():
    ax3.plot(x, y, 'o', color='#2980b9', markersize=22)
    ax3.text(x, y, nombre, ha='center', va='center', fontsize=7, fontweight='bold', color='white')
    ax3.annotate('', xy=(x*0.45, 0.1 + (y-0.1)*0.45), xytext=(x*0.75, 0.1 + (y-0.1)*0.75),
                 arrowprops=dict(arrowstyle='<->', color='#27ae60', lw=1.5))

ax3.text(0, -1.45, 'Cada sistema se conecta solo al hub',
         ha='center', fontsize=9, color='#27ae60', style='italic')

plt.tight_layout()
plt.show()

La diferencia visual es clara. En el patron punto a punto, cada nuevo sistema multiplica la complejidad exponencialmente. Con el ETL centralizado ganamos orden, pero el flujo es unidireccional: las correcciones no vuelven a las fuentes. El Data Hub resuelve ambos problemas: centraliza la integracion y permite que la informacion corregida fluya de vuelta a todos los sistemas.

Aplicado a la auditoria de Santa Clara: si existiera un Data Hub con MDM, habria una unica ficha maestra de cada paciente. Cuando farmacia detectara una nueva alergia, esa informacion se propagaria automaticamente al HIS y estaria disponible para laboratorio. Los problemas de nombres inconsistentes se resolverian porque el hub mantendria la version canonica.

### Diferencias clave entre Data Hub, Data Warehouse y Data Lake

Es facil confundir estos tres conceptos porque los tres implican "centralizar datos". La distincion esta en su proposito:

| Concepto | Proposito principal | Direccion del flujo | Dato que almacena |
|---|---|---|---|
| **Data Warehouse** | Analisis y reporting | Unidireccional (fuentes -> DW) | Datos historicos, transformados |
| **Data Lake** | Almacenamiento flexible | Unidireccional (fuentes -> lake) | Datos crudos, todos los formatos |
| **Data Hub** | Integracion y sincronizacion | Bidireccional (fuentes <-> hub) | Datos maestros, version canonica |

Un Data Hub no sustituye al warehouse ni al lake; los **complementa**. En una arquitectura madura, el Data Hub alimenta al warehouse (con datos limpios y consistentes) y al lake (con datos estandarizados). Es el "notario" que certifica la identidad de cada entidad antes de que los datos fluyan al resto de la arquitectura.

## 3. Gobierno del dato: el marco organizativo

Hemos visto la solucion tecnica (Data Hub + MDM). Pero la tecnologia sola no basta. Alguien tiene que decidir como se nombran los campos, quien puede modificar una ficha de paciente, que ocurre cuando dos sistemas discrepan, como se audita la calidad, y que consecuencias tiene el incumplimiento.

Eso es el **gobierno del dato** (data governance): el conjunto de politicas, procesos, roles y metricas que aseguran que los datos de una organizacion sean correctos, seguros, accesibles y utiles.

Es importante entender lo que el gobierno del dato **NO es**:

- **No es solo tecnologia**. Puedes tener las mejores herramientas de calidad de datos y seguir teniendo un caos si nadie define las reglas.
- **No es solo cumplimiento legal**. El RGPD es una parte, pero el gobierno va mucho mas alla: abarca calidad, accesibilidad, eficiencia operativa.
- **No es un proyecto con fecha de fin**. Es un proceso continuo que evoluciona con la organizacion.
- **No es responsabilidad exclusiva de TI**. Involucra a negocio, legal, clinica, direccion -- a toda la organizacion.

### Los cinco pilares del gobierno del dato

Cualquier marco de gobierno del dato se construye sobre cinco pilares fundamentales. Veamoslos aplicados a Santa Clara:

| Pilar | Que responde | Ejemplo en Santa Clara |
|---|---|---|
| **1. Propiedad** | Quien es responsable de cada dato | "El Servicio de Admision es propietario de los datos de identificacion del paciente" |
| **2. Calidad** | Como garantizamos que los datos sean correctos y completos | "Las alergias deben registrarse en todos los sistemas en las primeras 24h desde el ingreso" |
| **3. Seguridad** | Quien puede acceder a que datos y como se protegen | "Solo el equipo medico directo puede acceder al historial clinico completo; investigacion accede a datos anonimizados" |
| **4. Cumplimiento** | Que normativas debemos respetar y como demostramos que lo hacemos | "Los datos clinicos se retienen 5 anos tras el alta conforme al RGPD y a la normativa sanitaria" |
| **5. Ciclo de vida** | Como se crean, transforman, archivan y eliminan los datos | "Los datos de wearables se agregan tras 90 dias y los registros granulares se eliminan tras 1 ano" |

Estos cinco pilares son interdependientes. Sin propiedad clara, nadie se responsabiliza de la calidad. Sin calidad, la seguridad se aplica sobre datos erroneos. Sin cumplimiento, la organizacion se expone a sanciones. Sin gestion del ciclo de vida, el almacenamiento crece sin control y los datos obsoletos contaminan los analisis.

Observa la conexion con la sesion de etica del bloque anterior: los principios eticos (transparencia, equidad, responsabilidad) se **operativizan** a traves del gobierno del dato. La etica dice "debemos evitar sesgos"; el gobierno dice "el Data Steward de laboratorio debe verificar mensualmente que la distribucion demografica de las muestras es representativa".

## 4. Roles y responsabilidades del gobierno del dato

Un marco de gobierno no funciona si no hay personas concretas asignadas a responsabilidades concretas. Los roles clave son:

**Data Owner (Propietario del dato)**
Responsable de negocio que "posee" un dominio de datos. Decide quien accede, aprueba definiciones y es el responsable ultimo de la calidad. En Santa Clara, el Data Owner de los datos de pacientes seria el Director Medico o el responsable de Admision.

**Data Steward (Administrador del dato)**
Perfil operativo que implementa las politicas del Data Owner en el dia a dia. Monitoriza la calidad, gestiona incidencias, documenta metadatos. Es el "guardian cotidiano" del dato. En Santa Clara, cada servicio (urgencias, farmacia, laboratorio) tendria un Data Steward.

**Data Custodian (Custodio tecnico)**
Perfil de TI responsable de la infraestructura tecnica: almacenamiento, backups, seguridad de acceso, rendimiento. No decide QUE datos se guardan, sino COMO se guardan de forma segura y eficiente.

**DPO (Data Protection Officer)**
Rol exigido por el RGPD en organizaciones que tratan datos sensibles a gran escala. Supervisa el cumplimiento normativo, asesora sobre evaluaciones de impacto y actua como enlace con la autoridad de proteccion de datos.

**Comite de Datos (Data Governance Council)**
Organo colegiado que establece la estrategia, aprueba politicas transversales y resuelve conflictos entre dominios. Incluye representantes de negocio, TI, legal y direccion.

Vamos a modelar estos roles para Santa Clara y crear una herramienta sencilla que, dado un incidente de datos, identifique que rol debe actuar:

In [ ]:
# Roles de gobierno del dato en Santa Clara
roles_santa_clara = {
    'Data Owner': {
        'persona': 'Dra. Elena Vidal (Directora Medica)',
        'responsabilidad': 'Decide politicas de acceso y calidad de datos clinicos',
        'dominio': 'Datos de pacientes, historiales clinicos',
        'actua_ante': ['Definicion de nuevos campos', 'Aprobacion de accesos', 
                       'Conflictos entre servicios']
    },
    'Data Steward - Urgencias': {
        'persona': 'Carlos Ruiz (Coord. Enfermeria Urgencias)',
        'responsabilidad': 'Monitoriza calidad diaria de datos de triaje y urgencias',
        'dominio': 'Registros de urgencias, triaje',
        'actua_ante': ['Duplicados detectados', 'Campos incompletos',
                       'Discrepancias en codigos de diagnostico']
    },
    'Data Steward - Farmacia': {
        'persona': 'Laura Mendez (Farmaceutica adjunta)',
        'responsabilidad': 'Garantiza integridad del registro de dispensaciones y alergias',
        'dominio': 'Prescripciones, dispensaciones, alergias',
        'actua_ante': ['Alergias no sincronizadas', 'Medicamentos sin prescripcion',
                       'Errores de codificacion ATC']
    },
    'Data Custodian': {
        'persona': 'Miguel Torres (Responsable de Sistemas)',
        'responsabilidad': 'Mantiene la infraestructura tecnica y la seguridad de acceso',
        'dominio': 'Todos los sistemas (HIS, Farmacia, Laboratorio)',
        'actua_ante': ['Brecha de seguridad', 'Caida de sistema', 'Backup fallido',
                       'Solicitud de nuevo acceso tecnico']
    },
    'DPO': {
        'persona': 'Ana Jimenez (Delegada de Proteccion de Datos)',
        'responsabilidad': 'Supervisa cumplimiento RGPD y normativa sanitaria',
        'dominio': 'Transversal (todos los datos personales)',
        'actua_ante': ['Solicitud de acceso/rectificacion de un paciente',
                       'Incidente de seguridad con datos personales',
                       'Evaluacion de impacto de nuevo tratamiento de datos']
    }
}

print("=== Roles de gobierno del dato en Santa Clara ===\n")
for rol, info in roles_santa_clara.items():
    print(f"  {rol}")
    print(f"    Persona:         {info['persona']}")
    print(f"    Responsabilidad: {info['responsabilidad']}")
    print(f"    Dominio:         {info['dominio']}")
    print()

In [ ]:
# Funcion de respuesta a incidentes: dado un problema, quien actua?
incidentes = {
    'Duplicado de paciente detectado en HIS': {
        'responsable_primario': 'Data Steward - Urgencias',
        'notificar_a': ['Data Owner', 'Data Custodian'],
        'accion': 'Investigar origen del duplicado, fusionar registros, documentar causa raiz'
    },
    'Alergia registrada en farmacia pero no en HIS': {
        'responsable_primario': 'Data Steward - Farmacia',
        'notificar_a': ['Data Owner', 'Data Steward - Urgencias'],
        'accion': 'Verificar alergia con el paciente, actualizar HIS, revisar protocolo de sincronizacion'
    },
    'Paciente solicita acceso a todos sus datos clinicos': {
        'responsable_primario': 'DPO',
        'notificar_a': ['Data Owner', 'Data Custodian'],
        'accion': 'Tramitar solicitud ARCO en plazo legal (30 dias), recopilar datos de todos los sistemas'
    },
    'Brecha de seguridad: acceso no autorizado a historiales': {
        'responsable_primario': 'Data Custodian',
        'notificar_a': ['DPO', 'Data Owner', 'Comite de Datos'],
        'accion': 'Contener brecha, notificar a AEPD en 72h, investigar, informar a afectados si procede'
    },
    'Laboratorio usa formato de ID distinto al resto': {
        'responsable_primario': 'Data Owner',
        'notificar_a': ['Data Custodian', 'Data Steward - Urgencias', 'Data Steward - Farmacia'],
        'accion': 'Definir estandar unico de ID, encargar migracion a Data Custodian, verificar integridad'
    }
}

print("=== Protocolo de respuesta a incidentes de datos ===\n")
for incidente, protocolo in incidentes.items():
    print(f"  INCIDENTE: {incidente}")
    print(f"    Responsable primario: {protocolo['responsable_primario']}")
    print(f"    Notificar a:          {', '.join(protocolo['notificar_a'])}")
    print(f"    Accion:               {protocolo['accion']}")
    print()

Volvamos a la auditoria de la seccion 1. Con estos roles definidos, podemos asignar responsabilidades a cada hallazgo:

- **Nombres inconsistentes entre sistemas**: Responsabilidad del **Data Owner** (deberia haber definido un estandar de nomenclatura) y del **Data Custodian** (deberia haber implementado validaciones tecnicas).
- **Alergias no sincronizadas**: Responsabilidad de los **Data Stewards** de cada servicio (deberian tener un protocolo de actualizacion cruzada) y fallo de integracion que el **Data Custodian** deberia resolver tecnicamente (idealmente con un Data Hub).
- **Registro huerfano P-1099 en laboratorio**: Responsabilidad del **Data Steward** de laboratorio (deberia haber un control de integridad referencial) y del **Data Custodian** (deberia existir una validacion tecnica que impida crear registros sin alta previa en HIS).
- **Formato de ID distinto (PAC-1004)**: Fallo de gobernanza clasico. El **Data Owner** deberia haber definido un estandar unico, y el **Comite de Datos** deberia haberlo aprobado como politica transversal.

El patron es claro: los problemas tecnicos (integracion, formatos) y los problemas organizativos (roles, politicas) son dos caras de la misma moneda. No se resuelve uno sin el otro.

## 5. De la auditoria al plan de accion

Una auditoria no tiene valor si no se traduce en acciones concretas. Pero los recursos son limitados, asi que hay que priorizar. Una herramienta clasica para esto es la **matriz de impacto vs. esfuerzo**: posicionamos cada problema detectado segun la gravedad de su impacto (clinico, legal, operativo) y el esfuerzo necesario para resolverlo (tecnico, organizativo, economico).

Los problemas que caen en el cuadrante de alto impacto y bajo esfuerzo son los "quick wins" que debemos abordar primero. Los de alto impacto y alto esfuerzo son proyectos estrategicos que requieren planificacion. Los de bajo impacto pueden esperar.

In [ ]:
# Hallazgos de la auditoria posicionados en la matriz impacto vs esfuerzo
hallazgos = pd.DataFrame({
    'hallazgo': [
        'Alergias no sincronizadas',
        'Formato de ID inconsistente',
        'Nombres sin normalizar',
        'Registros huerfanos',
        'Lab no registra alergias',
        'Sin Data Hub / MDM',
        'Sin roles de gobierno',
        'Sin protocolo de alta unica'
    ],
    'impacto': [5, 3, 2, 3, 4, 5, 4, 4],    # 1=bajo, 5=critico
    'esfuerzo': [2, 2, 1, 2, 3, 5, 3, 3],    # 1=facil, 5=muy costoso
    'tipo': [
        'Clinico', 'Tecnico', 'Tecnico', 'Tecnico',
        'Clinico', 'Estrategico', 'Organizativo', 'Organizativo'
    ]
})

fig, ax = plt.subplots(figsize=(10, 7))

colores_tipo = {
    'Clinico': '#c0392b',
    'Tecnico': '#2980b9',
    'Estrategico': '#8e44ad',
    'Organizativo': '#e67e22'
}

for _, row in hallazgos.iterrows():
    color = colores_tipo[row['tipo']]
    ax.scatter(row['esfuerzo'], row['impacto'], c=color, s=200, zorder=5, edgecolors='white', linewidth=1.5)
    # Desplazar etiquetas para evitar solapamientos
    offset_x = 0.12
    offset_y = 0.12
    ax.annotate(row['hallazgo'], (row['esfuerzo'] + offset_x, row['impacto'] + offset_y),
                fontsize=8, ha='left')

# Cuadrantes
ax.axhline(y=3, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=3, color='gray', linestyle='--', alpha=0.5)

# Etiquetas de cuadrantes
ax.text(1.5, 4.7, 'QUICK WINS\n(prioridad alta)', ha='center', fontsize=10,
        fontweight='bold', color='#27ae60', alpha=0.7)
ax.text(4.2, 4.7, 'PROYECTOS\nESTRATEGICOS', ha='center', fontsize=10,
        fontweight='bold', color='#8e44ad', alpha=0.7)
ax.text(1.5, 1.5, 'MEJORAS\nINCREMENTALES', ha='center', fontsize=10,
        fontweight='bold', color='#2980b9', alpha=0.7)
ax.text(4.2, 1.5, 'BAJA\nPRIORIDAD', ha='center', fontsize=10,
        fontweight='bold', color='#999999', alpha=0.7)

ax.set_xlabel('Esfuerzo (1=facil, 5=muy costoso)', fontsize=11)
ax.set_ylabel('Impacto (1=bajo, 5=critico)', fontsize=11)
ax.set_title('Matriz de priorizacion -- Hallazgos de la auditoria de Santa Clara', fontsize=12)
ax.set_xlim(0.5, 5.5)
ax.set_ylim(0.5, 5.5)

# Leyenda
from matplotlib.patches import Patch
leyenda = [Patch(facecolor=c, label=t) for t, c in colores_tipo.items()]
ax.legend(handles=leyenda, title='Tipo de hallazgo', loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

La matriz nos da un mapa de accion claro:

**Quick wins (alto impacto, bajo esfuerzo):**
- Sincronizar alergias entre HIS y farmacia: es critico clinicamente y solo requiere establecer un protocolo y un cruce automatizado sencillo.
- Normalizar nombres: bajo impacto clinico pero facil de resolver con reglas de estandarizacion.
- Resolver registros huerfanos y estandarizar IDs: requiere una limpieza puntual y una regla de validacion.

**Proyectos estrategicos (alto impacto, alto esfuerzo):**
- Implementar un Data Hub con MDM: es la solucion de fondo, pero requiere inversion significativa en tecnologia, procesos y formacion.
- Definir roles de gobierno y protocolo de alta unica: requiere cambio organizativo y aprobacion de direccion.

**La leccion clave:** no hay que esperar a tener la solucion perfecta (Data Hub) para resolver problemas urgentes (alergias no sincronizadas). Se empieza por los quick wins mientras se planifican los proyectos estrategicos. El gobierno del dato es un camino, no un destino.

## 6. Reflexion y conexion con la siguiente sesion

Hoy hemos completado el arco que empezo en la sesion anterior: del "plano" (arquitectura) hemos pasado al "reglamento" (gobierno). Hemos visto que los problemas de datos en las organizaciones son siempre una combinacion de fallos tecnicos (falta de integracion) y organizativos (falta de roles, politicas y procesos).

**Preguntas para discusion en grupo:**

1. Pensad en vuestra experiencia (trabajo, practicas, proyectos de clase). Habeis visto alguna vez datos contradictorios entre dos sistemas o dos fuentes? Que consecuencias tuvo?

2. En Santa Clara, quien deberia ser el Data Owner de los datos de un paciente: el medico que le atiende, el servicio de admision, o la direccion del hospital? Argumentad vuestra posicion.

3. Imaginad que sois consultores contratados para implementar el gobierno del dato en una PYME de 50 empleados. Necesitan los cinco roles que hemos visto? Como simplificariais el modelo?

**Conexion con la siguiente sesion:**
Hoy hemos introducido los conceptos fundamentales del gobierno del dato: pilares, roles, y como responder a una auditoria. En la proxima sesion formalizaremos todo esto en un **marco de trabajo** completo: objetivos medibles, indicadores de calidad, y la estructura organizativa detallada de un programa de gobierno del dato.